# NOTE: Consider removing this step in the future and replace with code in model stream

In [1]:
import pandas as pd
import os
import numpy as np

# import global TDM functions
import sys
sys.path.insert(0, '../Resources/2-Python/global-functions')
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()

bhereth
C:\Users\bhereth\confidential-2023-utah-hts-db5335615978.json


# Get Home-Based Job percents from external process

In [2]:
wah_df = client.query("SELECT * FROM `wfrc-modeling-data.ext_rsg_wah_2023.control_total_work_at_home_2023`").to_dataframe()
wah_df

,_Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_GVED,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME
0,392023,39,2023,0.062,0.031,0.024,0.074,0.177,0.045,0.041,...,0.027,0.025,0.074,0.001,0.001,0.021,0.036,0.060,0.096,"Sanpete County, Utah"
1,252023,25,2023,0.038,0.043,0.162,0.136,0.217,0.206,0.271,...,0.015,0.020,0.015,0.005,0.005,0.023,0.012,0.159,0.170,"Kane County, Utah"
2,72023,7,2023,0.043,0.017,0.011,0.136,0.152,0.024,0.025,...,0.008,0.008,0.014,0.008,0.008,0.010,0.014,0.041,0.055,"Carbon County, Utah"
3,12023,1,2023,0.076,0.052,0.093,0.136,0.083,0.054,0.054,...,0.074,0.074,0.033,0.011,0.011,0.051,0.061,0.044,0.105,"Beaver County, Utah"
4,452023,45,2023,0.065,0.050,0.037,0.064,0.258,0.139,0.163,...,0.029,0.034,0.015,0.013,0.013,0.009,0.024,0.113,0.137,"Tooele County, Utah"
5,472023,47,2023,0.019,0.007,0.054,0.083,0.174,0.012,0.015,...,0.010,0.012,0.019,0.013,0.013,0.014,0.029,0.035,0.065,"Uintah County, Utah"
6,412023,41,2023,0.069,0.003,0.128,0.136,0.128,0.058,0.066,...,0.029,0.033,0.026,0.014,0.014,0.030,0.032,0.063,0.095,"Sevier County, Utah"
7,52023,5,2023,0.072,0.058,0.055,0.097,0.219,0.076,0.076,...,0.021,0.021,0.031,0.015,0.015,0.036,0.028,0.101,0.129,"Cache County, Utah"
8,352023,35,2023,0.106,0.061,0.101,0.155,0.279,0.126,0.121,...,0.024,0.023,0.028,0.016,0.016,0.016,0.028,0.149,0.177,"Salt Lake County, Utah"
9,292023,29,2023,0.083,0.033,0.023,0.136,0.269,0.044,0.039,...,0.015,0.013,0.058,0.021,0.021,0.024,0.031,0.091,0.122,"Morgan County, Utah"


In [3]:
# FOR INCOMPLETE WAH DATA, SET WAH VALUES TO ZERO FOR ALL FORECAST YEARS

# 1. Identify unique counties
unique_fips = wah_df["CO_FIPS"].unique()

# 2. Determine existing year range and build new years
max_year = wah_df["YEAR"].max()
new_years = list(range(max_year + 1, 2066))   # 2065 inclusive

# If nothing to add, stop here
if len(new_years) > 0:

    # 3. Create a MultiIndex of all combinations
    new_index = pd.MultiIndex.from_product(
        [unique_fips, new_years],
        names=["CO_FIPS", "YEAR"]
    )

    # 4. Build an empty dataframe with those keys
    # Zero-fill all other numeric columns
    new_rows = pd.DataFrame(index=new_index).reset_index()

    # Identify numeric columns except CO_FIPS & YEAR
    num_cols = wah_df.select_dtypes(include="number").columns.difference(["CO_FIPS", "YEAR"])

    # Fill numeric with zeros
    for col in num_cols:
        new_rows[col] = 0

    # Add missing non-numeric columns too (if any)
    other_cols = wah_df.columns.difference(["CO_FIPS", "YEAR"] + list(num_cols))
    for col in other_cols:
        new_rows[col] = None

    # 5. Combine original + new rows
    wah_full_df = pd.concat([wah_df, new_rows], ignore_index=True)

    # 6. Sort neatly
    wah_full_df = wah_full_df.sort_values(["CO_FIPS", "YEAR"]).reset_index(drop=True)

else:
    wah_full_df = wah_df.copy()

wah_full_df['_Index'] = wah_full_df['CO_FIPS'] * 10000 + wah_full_df['YEAR']

wah_full_df


,_Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_GVED,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME
0,12023,1,2023,0.076,0.052,0.093,0.136,0.083,0.054,0.054,...,0.074,0.074,0.033,0.011,0.011,0.051,0.061,0.044,0.105,"Beaver County, Utah"
1,12024,1,2024,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
2,12025,1,2025,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
3,12026,1,2026,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
4,12027,1,2027,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,10492061,1049,2061,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
1286,10492062,1049,2062,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
1287,10492063,1049,2063,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None
1288,10492064,1049,2064,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,None


In [4]:
# normalize WAH data to long format

# get list of HBJ columns
hbj_cols = [c for c in wah_full_df.columns if c.startswith("HBJ_")]

# filter to only necessary columns
wah_hbj_df = wah_full_df[['CO_FIPS', 'YEAR'] + hbj_cols]

# melt the HBJ columns to long format
wah_melted_df = (
    wah_hbj_df
      .melt(
          id_vars=[c for c in wah_hbj_df.columns if not c.startswith("HBJ_")],
          value_vars=hbj_cols,
          var_name="HBJFULL",
          value_name="HBJPCT"
      )
      .assign(
          TDM_SECTOR=lambda d: d["HBJFULL"].str.split("_").str[1]
      )
      .drop(columns=["HBJFULL"])
).sort_values(by=['CO_FIPS', 'YEAR', 'TDM_SECTOR']).reset_index(drop=True)

# select and reorder final columns
wah_melted_df = wah_melted_df[['CO_FIPS', 'YEAR', 'TDM_SECTOR', 'HBJPCT']].rename(columns={'TDM_SECTOR':'VARIABLE'})

wah_melted_df

,CO_FIPS,YEAR,VARIABLE,HBJPCT
0,1,2023,AGRI,0.011
1,1,2023,CONS,0.051
2,1,2023,FOOD,0.012
3,1,2023,GVED,0.074
4,1,2023,HLTH,0.074
...,...,...,...,...
14185,1049,2065,MING,0.000
14186,1049,2065,OFFI,0.000
14187,1049,2065,OTHR,0.000
14188,1049,2065,RETL,0.000


# Get employment data

In [5]:
tdm_employment_by_county_df = pd.read_csv('intermediate/split_emp.csv')
tdm_employment_by_county_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,738,1
1,1,2023,CONS,278,1
2,1,2023,FOOD,517,1
3,1,2023,GVED,810,1
4,1,2023,HLTH,160,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


# Join tables and calculate home-based jobs

In [6]:
tdm_hbj_df = pd.merge(
    tdm_employment_by_county_df,
    wah_melted_df,
    on=['CO_FIPS', 'YEAR', 'VARIABLE'],
    how='inner'
)
tdm_hbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS,HBJPCT
0,1,2023,AGRI,738,1,0.011
1,1,2023,CONS,278,1,0.051
2,1,2023,FOOD,517,1,0.012
3,1,2023,GVED,810,1,0.074
4,1,2023,HLTH,160,1,0.074
...,...,...,...,...,...,...
15604,9157,2065,MING,164,57,0.000
15605,9157,2065,OFFI,14202,57,0.000
15606,9157,2065,OTHR,54646,57,0.000
15607,9157,2065,RETL,20356,57,0.000


In [7]:
# do banker's rounding to calculate HBJ counts
tdm_hbj_df['HBJ'] = np.round(
    tdm_hbj_df['VALUE'] * tdm_hbj_df['HBJPCT']
).astype(int)

tdm_hbj_df['NONHBJ'] = (tdm_hbj_df['VALUE'] - tdm_hbj_df['HBJ']).astype(int)
tdm_hbj_df = tdm_hbj_df[['CO_FIPS_split','CO_FIPS','YEAR','VARIABLE','VALUE','HBJPCT','HBJ','NONHBJ']]
tdm_hbj_df

,CO_FIPS_split,CO_FIPS,YEAR,VARIABLE,VALUE,HBJPCT,HBJ,NONHBJ
0,1,1,2023,AGRI,738,0.011,8,730
1,1,1,2023,CONS,278,0.051,14,264
2,1,1,2023,FOOD,517,0.012,6,511
3,1,1,2023,GVED,810,0.074,60,750
4,1,1,2023,HLTH,160,0.074,12,148
...,...,...,...,...,...,...,...,...
15604,9157,57,2065,MING,164,0.000,0,164
15605,9157,57,2065,OFFI,14202,0.000,0,14202
15606,9157,57,2065,OTHR,54646,0.000,0,54646
15607,9157,57,2065,RETL,20356,0.000,0,20356


In [8]:
tdm_hbj_df.to_csv('intermediate/split_emp_with_hbj_detail.csv', index=False)

## RUN CHECKS

In [9]:
# 1. Before HBJ: totals from tdm_employment_by_county_df
before_df = (
    tdm_employment_by_county_df
      .groupby(['CO_FIPS_split', 'YEAR'], as_index=False)
      .agg(VALUE_BEFORE_HBJ=('VALUE', 'sum'))
)

# 2. After HBJ: totals from tdm_hbj_df
after_df = (
    tdm_hbj_df
      .groupby(['CO_FIPS_split', 'YEAR'], as_index=False)
      .agg(
          VALUE_AFTER_HBJ=('HBJ', 'sum'),
          NONHBJ_AFTER_HBJ=('NONHBJ', 'sum')
      )
      .assign(
          TOTAL_AFTER_HBJ=lambda d: d['VALUE_AFTER_HBJ'] + d['NONHBJ_AFTER_HBJ']
      )
)

# 3. Combine and compute difference
tdm_hbj_check_df = (
    before_df
      .merge(after_df, on=['CO_FIPS_split', 'YEAR'], how='outer')
      .assign(
          DIFF=lambda d: d['VALUE_BEFORE_HBJ'] - d['TOTAL_AFTER_HBJ']
      )
)

# show where differences exist
tdm_hbj_check_df[tdm_hbj_check_df['DIFF'] != 0]


,CO_FIPS_split,YEAR,VALUE_BEFORE_HBJ,VALUE_AFTER_HBJ,NONHBJ_AFTER_HBJ,TOTAL_AFTER_HBJ,DIFF


### ONLY 2023 DATA FOR NOW

In [10]:
if tdm_hbj_check_df[(tdm_hbj_check_df['DIFF'] != 0)].size == 0:
    print("HBJ calculation check passed: No differences found for 2023 data.")
else:
    print("else")

HBJ calculation check passed: No differences found for 2023 data.


# Change to long format

In [11]:
long_nonhbj_df = tdm_hbj_df[['CO_FIPS_split','YEAR','VARIABLE','NONHBJ','CO_FIPS']].rename(columns={'NONHBJ':'VALUE'})
long_nonhbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,730,1
1,1,2023,CONS,264,1
2,1,2023,FOOD,511,1
3,1,2023,GVED,750,1
4,1,2023,HLTH,148,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


In [12]:
long_hbj_df = tdm_hbj_df.groupby(['CO_FIPS_split','YEAR','CO_FIPS'], as_index=False).agg(VALUE=('HBJ','sum'))
long_hbj_df['VARIABLE'] = 'HBJ'
long_hbj_df = long_hbj_df[['CO_FIPS_split','YEAR','VARIABLE','VALUE','CO_FIPS']]
long_hbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,HBJ,244,1
1,1,2024,HBJ,0,1
2,1,2025,HBJ,0,1
3,1,2026,HBJ,0,1
4,1,2027,HBJ,0,1
...,...,...,...,...,...
1414,9157,2061,HBJ,0,57
1415,9157,2062,HBJ,0,57
1416,9157,2063,HBJ,0,57
1417,9157,2064,HBJ,0,57


In [13]:
long_df = pd.concat([long_hbj_df,long_nonhbj_df]).sort_values(['CO_FIPS_split','YEAR','VARIABLE'])
long_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,730,1
1,1,2023,CONS,264,1
2,1,2023,FOOD,511,1
3,1,2023,GVED,750,1
0,1,2023,HBJ,244,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


In [14]:
long_df.to_csv('intermediate/split_emp_with_hbj.csv', index=False)